# Any alarm? Two-hour history → next 10, 20 or 30 minutes

This is an independent **any-code alarm-start** experiment. It does not change the older icing-group notebooks.

**Run order:** check data → visualise → fit → freeze → optionally test. The prepared Parquets already exist; skip the Excel and alignment notebooks.

At 12:00, sensors from 10:00 through 11:50 produce three yes/no forecasts. A 20-minute yes means at least one recorded alarm starts in **[12:00, 12:20)**. A start exactly at 12:20 belongs to a later window. Existing active alarms are not excluded. Alarm columns are targets/audit fields, never predictors.

Install the project's `requirements.txt` in your selected notebook kernel. All experiment settings live in `configs/short_horizon_alarm.json`; paths are relative to the repository, not a teammate's drive.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'configs' / 'short_horizon_alarm.json').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from inside the Git repository.')
sys.path.insert(0, str(REPO / 'src'))
from wt_predmaint.short_horizon import load_config
CFG = load_config(REPO / 'configs' / 'short_horizon_alarm.json')
print('Repository:', REPO)


## 1. Inspect the configuration

Defaults pool all 21 physical turbines, train through 2022, validate in 2023, and reserve 2024–2025 for final testing. 2019 is absent. These years have been used in earlier team experiments, so this is a chronological holdout, not a historically untouched dataset.

The initial run uniformly samples up to 120,000 training windows and 50,000 each for validation and test. No class-based sampling is applied to evaluation. Set caps to `None` for all eligible rows only if you have enough RAM/time. Sampling and class rates are saved in `sampling.json`.


In [ ]:
display(pd.Series(CFG, name='setting').to_frame())
# Optional changes for this run (also saved in configuration.json):
# CFG['turbine_ids'] = ['2304511']
# CFG['max_train_rows'] = None


## 2. Validate existing preprocessing and construct windows

The source is already aligned to a 10-minute grid. This step validates it, masks missing source readings, and builds mean/std/min/max/last/change sensor summaries over exactly the preceding 12 bins. A row needs all 12 source intervals observed; individual absent sensors remain missing until training-only imputation.

Rows whose history or forecast extends beyond the current year or split are excluded. No future values, alarm fields, backward filling, random time split or cross-turbine rolling is used. No large feature cache is written.


In [ ]:
from wt_predmaint.short_horizon import development, fit_models, evaluate_test, new_run
OUT = new_run(REPO, CFG)
prepared = development(REPO, CFG, OUT)
display(pd.read_csv(OUT / 'data_quality.csv'))
display(pd.DataFrame({name: {'eligible_population': info['eligible_population'], 'sample_rows': info['sample_rows']}
                      for name, info in prepared['sampling'].items()}).T)
print('Run output:', OUT)


## 3. See what is happening before training

Coverage shows missing years/turbine observations. Missingness is computed from training years. Alarm prevalence uses all eligible development windows, not the training sample. The sensor trace preserves gaps and marks new starts of any code.


In [ ]:
for filename in ['01_coverage_missingness.png', '02_alarm_prevalence.png', '03_sensors_and_alarm_starts.png']:
    display(Image(filename=str(OUT / filename)))


## 4. Train three independent horizons

Each horizon compares logistic regression (median imputation + scaling), random forest, and histogram-based XGBoost, with a constant training-prevalence baseline. All transformations and class weights are fitted on training rows only.

For each model, validation chooses the threshold maximising F1. Validation average precision chooses the winning model per horizon. Saved scores are model outputs, **not guaranteed calibrated probabilities**. Window-level recall is not the percentage of independent alarm episodes detected.


In [ ]:
validation_metrics = fit_models(CFG, prepared)
display(validation_metrics)
display(pd.DataFrame(json.loads((OUT / 'frozen_selection.json').read_text())).T)
for filename in ['04_validation_precision_recall.png', '05_validation_confusion_matrices.png']:
    display(Image(filename=str(OUT / filename)))
if (OUT / '06_validation_forecast_timeline.png').is_file():
    display(Image(filename=str(OUT / '06_validation_forecast_timeline.png')))


## 5. Final test after freezing model choices and thresholds

The test step is opt-in to avoid accidental repeated examination. For a final run, set `RUN_TEST = True`. It opens 2024–2025 only after all fits and selections finish. Read saved results if this run has already been tested; do not use test metrics to retune the models.


In [ ]:
RUN_TEST = False
if (OUT / 'test_metrics.csv').is_file():
    display(pd.read_csv(OUT / 'test_metrics.csv'))
elif RUN_TEST:
    test_metrics = evaluate_test(REPO, CFG, prepared)
    display(test_metrics)
else:
    print('Test not run. Set RUN_TEST=True after reviewing/finalising development settings.')


Models, frozen thresholds, confusion matrices, PR curves, feature importance, sampled predictions and configuration are saved in `OUT`. Open the next notebook to review completed runs without retraining.
